# Seaborn

## Preliminary Note on the Data

Since the data is large (1.5 GB once loaded into a dataframe), we'll define one or more functions per exercise to create our charts.

This way, when each function returns, the temporary variables it used are deleted and the memory can be reused.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas
import seaborn as sns

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, between 2014 and 2022.

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s dvf

*You can load the data with the [`pandas.read_parquet`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html) function.*

In [ ]:
# Your code here

*Once the data is available in a pandas dataframe, create a `date` column from the `annee_mutation`, `mois_mutation` & `jour_mutation` columns and the [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) function, then drop these columns to save memory.*

In [ ]:
# Your code here

*Finally, display the memory size and the column types of the dataframe.*

In [ ]:
# Your code here

### Solution

In [ ]:
df = pandas.read_parquet("dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"], inplace=True)

In [ ]:
# For readability, keep the 0 in front of single-digit departments
replacedict = {
  "code_departement": {
    x: (x if len(x) > 1 else "0" + x) for x in df.code_departement.unique()
  }
}
df.replace(replacedict, inplace=True)

In [ ]:
df.info()

## Number of Transactions per Year

To start, we'll count the number of transactions (rows of the dataset) per year.

*Use the [`seaborn.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html) function and the [`dt.year`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html) attribute of the `date` column to create a count plot.*

In [ ]:
def transactions_by_year() -> None:
  pass  # Your code here


transactions_by_year()

### Solution

In [ ]:
def transactions_by_year() -> None:
  fig, ax = plt.subplots()
  sns.countplot(x=df.date.dt.year, ax=ax)
  sns.despine(fig)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(places=1))
  fig.show()


transactions_by_year()

## Total Property Value by Department

In this exercise, we'll group our data by the values of a column (here the department of the transaction) and display the total property value exchanged per group.

- *Group the data by department with the [`pandas.DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) function.*
- *Use the [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function to display this grouped data.*

In [ ]:
def value_by_department() -> None:
  pass  # Your code here


value_by_department()

### Solution

In [ ]:
def value_by_department() -> None:
  data = (
    df.groupby(df.code_departement).valeur_fonciere.sum().sort_values(ascending=False)
  )

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Total transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_department()

## Total Property Value by Region and by Year

In this part, we'll study how to break down a categorical variable by another variable: we'll produce a chart of the total property value exchanged by region, and this value will itself be broken down by year.

To do so, let's start by fetching a dataset that will let us group departments into regions:

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

The following function returns a pandas dataframe that makes it easy to join the region to our working dataframe.

In [ ]:
def get_regions_df() -> pandas.DataFrame:
  df = pandas.read_csv(
    "regions.csv",
    index_col="num_dep",
    usecols=["num_dep", "region_name"],
    dtype=dict(region_name="category"),
  )
  df.set_index(df.index.str.lstrip("0"), inplace=True)
  return df

- *Add a column containing the region name to your working dataframe, with the [`pandas.DataFrame.join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) function.*
- *Group the data by region then by year with the [`pandas.DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) function.*
- *Use the [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function to display this grouped data. You can refer to the part of the [seaborn tutorial](https://seaborn.pydata.org/tutorial/introduction.html#plots-for-categorical-data) about complex categorical data.*

In [ ]:
def value_by_region_by_year() -> None:
  pass  # Your code here


value_by_region_by_year()

### Solution

In [ ]:
def value_by_region_by_year() -> None:
  joined = df.join(get_regions_df(), on="code_departement")
  sort_index = (
    joined.groupby(joined.region_name)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .index
  )
  data = (
    joined.groupby([joined.region_name, joined.date.dt.year])
    .valeur_fonciere.sum()
    .reset_index()
  )

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(
    data, x="region_name", y="valeur_fonciere", hue="date", order=sort_index, ax=ax
  )
  sns.despine(fig)
  ax.set_title("Total transaction amount by region and by year")
  ax.set_xlabel("Region")
  plt.xticks(rotation=45)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_region_by_year()

## House and Apartment Prices

In this exercise, we'll compute the price per square meter of apartments and houses separately, and display how these prices evolve over the period considered.

- *For apartments, filter the working dataframe to keep only the rows where:*
    - *`nombre_appartements`, `valeur_fonciere` & `surface_reelle_bati_appartements` are strictly positive.*
    - *`nombre_maisons` is zero.*
- *Likewise, for houses, filter the working dataframe to keep only the rows where:*
    - *`nombre_maisons`, `valeur_fonciere` & `surface_reelle_bati_maisons` are strictly positive.*
    - *`nombre_appartements` is zero.*
- *Use this filtered data to compute the price per square meter for each month available in the dataset.*
- *Display the result with the [seaborn.lineplot](https://seaborn.pydata.org/generated/seaborn.lineplot.html) function.*

In [ ]:
def houses_apartments_prices() -> None:
  pass  # Your code here


houses_apartments_prices()

### Solution

In [ ]:
def houses_apartments_prices() -> None:
  mask_apartments = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  mask_houses = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )

  def get_prices(
    df: pandas.DataFrame, mask: pandas.Series, area_column: str
  ) -> pandas.Series:
    grouped = df[mask].groupby(df["date"].dt.to_period("M"))
    return grouped.valeur_fonciere.sum() / grouped[area_column].sum()

  data = pandas.concat(
    [
      get_prices(df, mask_houses, "surface_reelle_bati_maisons").rename("houses"),
      get_prices(df, mask_apartments, "surface_reelle_bati_appartements").rename(
        "apartments"
      ),
    ],
    axis=1,
  )
  data.reset_index(inplace=True)
  data = data.melt(id_vars=["date"], value_vars=["houses", "apartments"])
  data.date = data.date.astype("str")

  fig, ax = plt.subplots()
  sns.lineplot(data, x="date", y="value", hue="variable", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Average value")
  ax.set_title("Price per square meter")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(
    matplotlib.ticker.EngFormatter(unit="€/m²", places=1)
  )
  fig.show()


houses_apartments_prices()

## Correlations Between Variables

We'll now study the linear correlations with the property value, and more generally between variables.

- *Compute the linear correlations of all variables with `valeur_fonciere` with the [`pandas.DataFrame.corrwith`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html) function.*
- *Sort the variables from the most correlated (or anti-correlated) to the least correlated with `valeur_fonciere`. You can use [`pandas.Series.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.sort_values.html) and its `key` argument, combined with the [`abs`](https://docs.python.org/3/library/functions.html#abs) function.*
- *Compute the correlation matrix of these variables with the [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) function.*
- *Display this matrix with the [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) function.*

In [ ]:
def corrs() -> None:
  pass  # Your code here


corrs()

### Solution

In [ ]:
def corrs() -> None:
  corrs_to_price = (
    df.select_dtypes(include=["number"])
    .corrwith(df.valeur_fonciere)
    .sort_values(ascending=False, key=abs)
  )
  corrs = df.loc[:, corrs_to_price.index].corr()

  fig, ax = plt.subplots(figsize=(15, 10))
  sns.heatmap(corrs, vmin=-1, vmax=1, cmap="vlag", ax=ax)
  ax.set_title("Correlations between variables sorted by correlation to the price")
  fig.show()


corrs()